In [23]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import os


In [39]:
# =============================================================================
# LOAD YAML CONFIG
# =============================================================================

CONFIG_PATH = Path(os.environ.get("CONFIG_PATH", "esaf_config.yaml"))

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

YEAR_START = CFG["years"]["start"]
YEAR_END = CFG["years"]["end"]
REFERENCE_YEAR = CFG["years"]["reference_year"]
YEARS = np.arange(YEAR_START, YEAR_END + 1)

REF_CFG = CFG["refinery_inputs"]
JET_CFG = CFG["refinery_inputs"]
ESAF_CFG = CFG["esaf_parameters"]
DEMAND_CFG = CFG["demand_scenarios"]
OUT_CFG = CFG["output"]
CHECKS = CFG["checks"]

OUT_DIR = Path(OUT_CFG["base_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

H2_INTENSITY_ESAF = ESAF_CFG["h2_intensity_tH2_per_tSAF"]



In [40]:
# =============================================================================
# FUNCTIONS
# =============================================================================

def read_refinery(path: Path) -> pd.DataFrame:
    """
    Read refinery file using YAML schema.
    """

    df = pd.read_csv(path)

    schema = REF_CFG["schema"]

    df = df.rename(columns={
        schema["country_column"]: "country",
        schema["year_column"]: "year",
        schema["unit_column"]: "unit",
        schema["feed_column"]: "unit_feed",
    })

    required = {"country", "year", "unit", "unit_feed"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["unit_feed"] = pd.to_numeric(df["unit_feed"], errors="coerce").fillna(0.0)

    return df[df["year"].isin(YEARS)]


In [42]:
# -----------------------------------------------------------------------------
# Jet fossil reconstruction (ANTI DOUBLE COUNTING)
# -----------------------------------------------------------------------------

def compute_jet_fossil(df: pd.DataFrame) -> pd.DataFrame:

    g = df.groupby(["country", "year", "unit"], as_index=False)["unit_feed"].sum()

    units = JET_CFG["jet_chain_units"]
    yields_cfg = JET_CFG["jet_yields"]
    method = JET_CFG["jet_reconstruction_method"]

    # ------------------------------
    # Method 1: prefer Kero Hydrotreater
    # ------------------------------
    if method == "prefer_kero_hydrotreater":

        kht_name = units["kero_hydrotreater"]
        kht = g[g["unit"] == kht_name]

        if not kht.empty:
            out = (
                kht.groupby(["country", "year"], as_index=False)["unit_feed"]
                .sum()
                .rename(columns={"unit_feed": "jet_fossil_kt_per_yr"})
            )
            return out

    # ------------------------------
    # Fallback: hydrocracker yields
    # ------------------------------
    parts = []

    for key in ["vgo_hydrocracker", "residue_hydrocracker"]:
        unit_name = units[key]
        yield_coeff = yields_cfg[key]

        tmp = g[g["unit"] == unit_name].copy()
        if tmp.empty:
            continue

        tmp["jet_part"] = yield_coeff * tmp["unit_feed"]
        parts.append(
            tmp.groupby(["country", "year"], as_index=False)["jet_part"].sum()
        )

    if not parts:
        raise ValueError("Jet reconstruction impossible (no valid units found).")

    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on=["country", "year"], how="outer", suffixes=("", "_r")).fillna(0.0)
        out["jet_part"] += out.pop("jet_part_r")

    return out.rename(columns={"jet_part": "jet_fossil_kt_per_yr"})


In [43]:
# -----------------------------------------------------------------------------
# eSAF substitution
# -----------------------------------------------------------------------------

def build_esaf_scenario(jet_fossil_df: pd.DataFrame, growth_rate: float) -> pd.DataFrame:

    rows = []

    for country, g in jet_fossil_df.groupby("country"):

        g = g.set_index("year").sort_index()

        if REFERENCE_YEAR not in g.index:
            continue

        jet_reference = g.loc[REFERENCE_YEAR, "jet_fossil_kt_per_yr"]

        for year in YEARS:

            if year not in g.index:
                continue

            jet_fossil = g.loc[year, "jet_fossil_kt_per_yr"]
            jet_total = jet_reference * ((1 + growth_rate) ** (year - REFERENCE_YEAR))
            jet_fossil_adj = min(jet_fossil, jet_total)
            jet_esaf = jet_total - jet_fossil_adj
            h2_esaf = jet_esaf * H2_INTENSITY_ESAF

            rows.append({
                "country": country,
                "year": year,
                "jet_fossil_kt_per_yr": jet_fossil_adj,
                "jet_total_kt_per_yr": jet_total,
                "jet_esaf_kt_per_yr": jet_esaf,
                "h2_esaf_kt_per_yr": h2_esaf,
            })

    return pd.DataFrame(rows)


In [44]:

# -----------------------------------------------------------------------------
# Checks
# -----------------------------------------------------------------------------

def run_checks(df: pd.DataFrame):

    if CHECKS["non_negative_flows"]:
        if (df[[
            "jet_fossil_kt_per_yr",
            "jet_total_kt_per_yr",
            "jet_esaf_kt_per_yr",
            "h2_esaf_kt_per_yr"
        ]] < -1e-9).any().any():
            raise ValueError("Negative flows detected.")

    if CHECKS["enforce_mass_balance"]:
        diff = df["jet_total_kt_per_yr"] - (
            df["jet_fossil_kt_per_yr"] + df["jet_esaf_kt_per_yr"]
        )
        if (diff.abs() > 1e-6).any():
            raise ValueError("Mass balance violation detected.")

    if CHECKS["full_year_coverage"]:
        expected = len(YEARS)
        for country, g in df.groupby("country"):
            if len(g["year"].unique()) != expected:
                raise ValueError(f"Incomplete year coverage for {country}")



In [46]:

# =============================================================================
# MAIN
# =============================================================================

for ref_name, ref_info in REF_CFG["scenarios"].items():

    print(f"\nProcessing refinery scenario: {ref_name}")

    df_ref = read_refinery(Path(ref_info["path"]))
    jet_fossil_df = compute_jet_fossil(df_ref)

    for demand_name, demand_info in DEMAND_CFG.items():

        growth = demand_info["annual_growth_rate"]
        scenario_name = f"{demand_name}_{ref_name}"

        print(f"  → Building eSAF scenario: {scenario_name}")

        df_esaf = build_esaf_scenario(jet_fossil_df, growth)

        run_checks(df_esaf)

        out_path = OUT_DIR / scenario_name
        out_path.mkdir(parents=True, exist_ok=True)

        df_esaf.to_csv(out_path / OUT_CFG["output_file_name"], index=False)

print("\nAll eSAF substitution scenarios successfully written.")


Processing refinery scenario: more-molecule
  → Building eSAF scenario: constant-demand_more-molecule
  → Building eSAF scenario: growth-3pct_more-molecule
  → Building eSAF scenario: decline-1pct_more-molecule

Processing refinery scenario: max-electron
  → Building eSAF scenario: constant-demand_max-electron
  → Building eSAF scenario: growth-3pct_max-electron
  → Building eSAF scenario: decline-1pct_max-electron

All eSAF substitution scenarios successfully written.
